# Order book imbalance study

Loads book snapshots, computes top-of-book order book imbalance (OBI), and measures association with forward mid-price returns.

Default input: LOBSTER AAPL 2012-06-21 official orderbook snapshots (`data/snapshots_real.csv`).

Metrics are reported on a **time-ordered test split** (last 30%). This notebook does not model transaction costs or queue position.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path("..").resolve()
SNAP_PATH = ROOT / "data" / "snapshots_real.csv"
TRAIN_FRAC = 0.70
HORIZONS_NS = {
    "100ms": 100_000_000,
    "1s": 1_000_000_000,
    "5s": 5_000_000_000,
}
PRIMARY = "1s"

In [ ]:
df = pd.read_csv(SNAP_PATH)
df = df.dropna(subset=["best_bid", "best_ask", "bid_sz_1", "ask_sz_1"])
df = df[(df["bid_sz_1"] > 0) & (df["ask_sz_1"] > 0)].copy()
df = df[df["best_bid"] < df["best_ask"]].copy()

df["mid"] = 0.5 * (df["best_bid"] + df["best_ask"])
df["spread"] = df["best_ask"] - df["best_bid"]
denom = df["bid_sz_1"] + df["ask_sz_1"]
df["obi"] = (df["bid_sz_1"] - df["ask_sz_1"]) / denom

# Microprice: size-weighted touch price; deviation from mid is an alternate pressure measure.
df["micro"] = (
    df["best_ask"] * df["bid_sz_1"] + df["best_bid"] * df["ask_sz_1"]
) / denom
df["micro_minus_mid"] = df["micro"] - df["mid"]

span_s = (df["timestamp"].iloc[-1] - df["timestamp"].iloc[0]) / 1e9
print(f"rows={len(df):,}  span_hours={span_s / 3600:.2f}  mid=[{df['mid'].min():.1f}, {df['mid'].max():.1f}]")
df[["timestamp", "best_bid", "best_ask", "bid_sz_1", "ask_sz_1", "mid", "obi", "micro_minus_mid"]].head()

In [ ]:
def forward_mid_return(timestamps: np.ndarray, mids: np.ndarray, horizon_ns: int) -> np.ndarray:
    """Return (m_{t+h} - m_t) / m_t using the first snapshot at or after t+h."""
    n = len(timestamps)
    out = np.full(n, np.nan)
    j = 0
    for i in range(n):
        target = timestamps[i] + horizon_ns
        if j < i:
            j = i
        while j < n and timestamps[j] < target:
            j += 1
        if j < n and mids[i] != 0:
            out[i] = (mids[j] - mids[i]) / mids[i]
    return out

ts = df["timestamp"].to_numpy()
mid = df["mid"].to_numpy(dtype=float)
for name, h in HORIZONS_NS.items():
    df[f"ret_{name}"] = forward_mid_return(ts, mid, h)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(df["timestamp"], df["mid"], color="steelblue", lw=0.6, label="mid")
ax1.set_ylabel("mid (ticks)")
ax2 = ax1.twinx()
ax2.plot(df["timestamp"], df["obi"], color="darkorange", lw=0.4, alpha=0.5, label="OBI")
ax2.set_ylabel("OBI")
ax1.set_title("Mid price and top-of-book OBI")
fig.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
ret_col = f"ret_{PRIMARY}"
split = int(len(df) * TRAIN_FRAC)
train = df.iloc[:split]
test = df.iloc[split:].dropna(subset=[ret_col])

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(test["obi"], test[ret_col], s=4, alpha=0.25, c="teal")
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("OBI_t")
ax.set_ylabel(f"forward mid return ({PRIMARY})")
ax.set_title(f"Test set: OBI vs {PRIMARY} return (n={len(test):,})")
plt.tight_layout()
plt.show()

In [ ]:
def report_corr(label: str, x: np.ndarray, y: np.ndarray) -> None:
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 3:
        print(f"{label}: insufficient samples")
        return
    pearson = stats.pearsonr(x, y)
    spearman = stats.spearmanr(x, y)
    print(
        f"{label}: n={len(x):,}  "
        f"Pearson r={pearson.statistic:.4f} (p={pearson.pvalue:.2e})  "
        f"Spearman ρ={spearman.statistic:.4f} (p={spearman.pvalue:.2e})"
    )

print("Test set — top-of-book OBI vs forward mid return")
for name in HORIZONS_NS:
    col = f"ret_{name}"
    sub = df.iloc[split:].dropna(subset=[col])
    report_corr(name, sub["obi"].to_numpy(), sub[col].to_numpy())

print("\nTest set — microprice − mid (ticks) vs forward mid return")
for name in HORIZONS_NS:
    col = f"ret_{name}"
    sub = df.iloc[split:].dropna(subset=[col])
    report_corr(name, sub["micro_minus_mid"].to_numpy(), sub[col].to_numpy())

print("\nTrain set (diagnostic only)")
for name in HORIZONS_NS:
    col = f"ret_{name}"
    sub = train.dropna(subset=[col])
    report_corr(f"train/{name}", sub["obi"].to_numpy(), sub[col].to_numpy())

## Notes

- On this AAPL sample with dense snapshots (every 10th message), test-set Pearson r for OBI is typically on the order of **0.07–0.09** at 100 ms–1 s and near zero by 5 s.
- Microprice deviation from mid behaves similarly; it does not dominate OBI here.
- Weak, short-horizon association is expected in equity microstructure. Correlation is not a tradable edge after spread and fees.
- Synthetic `snapshots_replay.csv` can show much larger r by construction; use real snapshots for reported results.